# 🏫 Teacher Workspace Agent — SFT → GRPO Training

**Stage 1 — SFT:** Distil Qwen 72B expert trajectories into Qwen3-4B via supervised fine-tuning.
**Stage 2 — GRPO:** Continue RL training with environment reward signal.

| Model | Role |
|---|---|
| `Qwen/Qwen2.5-72B-Instruct` | Teacher — generates expert demos via HF API |
| `unsloth/Qwen3-4B-unsloth-bnb-4bit` | Student — trained locally on T4 |

**Runtime:** T4 GPU | **Est. total time:** ~90 min

## 1. Install Dependencies

In [ ]:
!pip install unsloth -q
!pip install trl==0.12.2 -q
!pip install 'openenv-core[core]>=0.2.2' -q
!pip install wandb huggingface_hub matplotlib -q
print('Done')

## 2. Clone & Install Environment

In [ ]:
!git clone https://github.com/amydosomething/teacher-workspace-env.git
%cd teacher-workspace-env
!pip install -e . -q
print('Environment installed')

## 3. Imports & GPU Check

In [ ]:
import sys, os, json, re, time, random, torch
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Dict, Any, Optional

sys.path.insert(0, '/content/teacher-workspace-env')
from server.teacher_workspace_env_environment import TeacherWorkspaceEnvironment
from models import TeacherAction
from server.tasks import get_task_names

TASK_NAMES = get_task_names()
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Tasks   : {TASK_NAMES}')

## 4. WandB Login

In [ ]:
import wandb
wandb.login()  # paste your WandB API key when prompted
wandb.init(project='teacher-workspace-sft-grpo', name='qwen3-4b-sft-grpo-run1')

## 5. Shared Prompt Helpers

In [ ]:
SYSTEM_PROMPT = """You are a teacher assistant controlling Google Workspace tools.

OUTPUT RULES - non-negotiable:
- Output EXACTLY ONE JSON object per response, nothing else.
- Format: {"tool_name": "<name>", "params": {<key>: <value>}}
- No markdown, no explanation, no prose before or after the JSON.
- ONE action per response. Never output multiple actions.

CALCULATION RULES:
- AVERAGE of n values = (v1 + v2 + v3) / n, rounded to exactly 2 decimal places.
- Always compute step by step: first sum, then divide.

STOPPING RULES:
- Only perform actions explicitly required by the task prompt.
- Once all sub-tasks are complete, output: {"tool_name": "__done__", "params": {}}
- Never repeat a completed step.
- Never create labels or send emails not mentioned in the task.

AVAILABLE TOOLS:
Classroom (read) : list_classrooms, get_classroom, list_announcements
Classroom (write): create_classroom, create_announcement
Sheets (read)    : list_sheets, get_cells
Sheets (write)   : create_sheet, update_cell, add_note, set_formula, sort_range
Gmail (read)     : list_inbox, read_mail, search_mail
Gmail (write)    : send_mail, star_mail, create_label, assign_label
Calendar (read)  : list_events
Calendar (write) : create_event, create_meet_event

BEHAVIOUR RULES:
1. Complete sub-tasks in order, one action at a time.
2. Never repeat a successful action.
3. Use EXACT email addresses from the students/parents lists.
4. set_formula format: AVERAGE(C2,D2,E2) — uppercase, no spaces.
5. assign_label: use mail_id from SENT list, not inbox.
6. create_label BEFORE assign_label.
7. create_meet_event only for students where (C+D+E)/3 < 60.
8. add_note not update_cell for notes column.
"""

def obs_to_prompt(obs, step: int, history: List[str]) -> str:
    sheets_str = ''
    for name, data in obs.sheets.items():
        cells = data.get('cells', {})
        sheets_str += f'\n{name}:\n'
        for k, v in list(cells.items())[:35]:
            sheets_str += f'  {k}={v}\n'
        notes = data.get('notes', {})
        if notes:
            sheets_str += f'  Notes: {notes}\n'
        formulas = data.get('formulas', {})
        if formulas:
            sheets_str += f'  Formulas set: {list(formulas.keys())}\n'

    sent_str = ''
    for m in obs.sent[-15:]:
        sent_str += f"  mail_id={m['mail_id']} to={m['to']} subject={m.get('subject','')[:35]} labels={m.get('labels',[])}\n"

    meet_str = ''
    for evt in obs.calendar:
        if evt.get('meet_link'):
            meet_str += f"  MEET CREATED: '{evt['title']}' participants={evt['participants']}\n"

    students_str = '\n'.join(f"  {s['name']} email={s['email']} id={s['id']}" for s in obs.students)
    parents_str  = '\n'.join(f"  {p['name']} email={p['email']} student_id={p['student_id']}" for p in obs.parents)

    feedback = ''
    if obs.error:
        feedback = f'\n!! LAST ACTION FAILED: {obs.error}\n   Change your approach.\n'
    elif obs.result:
        feedback = f'\n++ Last result: {json.dumps(obs.result)[:150]}\n'

    cls_str = ''
    for cid, c in obs.classrooms.items():
        cls_str += f"  {cid}: {c['name']} announcements={len(c.get('announcements',[]))}\n"

    inbox_str = ''
    for m in obs.inbox:
        inbox_str += f"  {m['mail_id']} from={m.get('from_name','?')} starred={m.get('starred')}\n"

    return (
        f'TASK:\n{obs.task_prompt}\n\n'
        f'STEP {step}\n\n'
        f'HISTORY (last 15):\n' + ('\n'.join(history[-15:]) or '  (none)') + '\n'
        f'{feedback}\n'
        f'CLASSROOMS:\n{cls_str or "  (none)"}\n'
        f'MEET EVENTS CREATED (do NOT recreate):\n{meet_str or "  (none)"}\n'
        f'GRADEBOOK DATA:\n{sheets_str}\n'
        f'INBOX:\n{inbox_str or "  (empty)"}\n'
        f'SENT EMAILS (use mail_id for assign_label):\n{sent_str or "  (none)"}\n'
        f'STUDENTS (exact emails):\n{students_str}\n'
        f'PARENTS (exact emails):\n{parents_str}\n'
        f'LABELS: {obs.labels}\n\n'
        f'Respond with ONE JSON object:'
    )

def parse_action(text: str) -> Optional[Dict]:
    text = re.sub(r'```(?:json)?', '', text).strip().replace('```', '').strip()
    for fn in [
        lambda t: json.loads(t),
        lambda t: json.loads(t[t.find('{'):t.rfind('}')+1]),
    ]:
        try:
            d = fn(text)
            if 'tool_name' in d:
                return d
        except:
            pass
    return None

MAX_STEPS = {'setup_new_course': 12, 'grade_and_notify': 25, 'end_of_semester': 35}
print('Prompt helpers ready')

## 6. Generate SFT Dataset from Qwen 72B (via HF Inference API)

Runs `Qwen/Qwen2.5-72B-Instruct` via HF Inference API — no local VRAM needed.
Only trajectories with `grade() >= 0.6` are kept as training examples.
Each step becomes a `(system, user, assistant)` triplet.
Output saved to `sft_data.jsonl`.

In [ ]:
from huggingface_hub import InferenceClient

# ── Config ──────────────────────────────────────────────────────────────────
HF_TOKEN        = 'hf_YOUR_TOKEN_HERE'   # <- paste your HF token
TEACHER_MODEL   = 'Qwen/Qwen2.5-72B-Instruct'
EPISODES_PER_TASK  = 8    # 8 x 3 tasks = 24 episodes; filtered down to ~12-15
GRADE_THRESHOLD    = 0.60 # only keep trajectories where grade() >= this
MAX_API_RETRIES    = 5
SFT_DATA_PATH      = 'sft_data.jsonl'

api_client = InferenceClient(token=HF_TOKEN)

def call_api(messages: list, temperature: float = 0.3) -> Optional[str]:
    for attempt in range(MAX_API_RETRIES):
        try:
            resp = api_client.chat_completion(
                model=TEACHER_MODEL,
                messages=messages,
                max_tokens=300,
                temperature=temperature,
            )
            return resp.choices[0].message.content
        except Exception as e:
            wait = (2 ** (attempt + 1)) * (2 if '429' in str(e) else 1)
            print(f'    API attempt {attempt+1}/{MAX_API_RETRIES} failed, waiting {wait}s: {e}')
            time.sleep(wait)
    return None

def run_teacher_episode(task_name: str):
    """
    Run one episode with Qwen 72B teacher.
    Returns (trajectory_pairs, grade_score).
    trajectory_pairs: list of (user_prompt_str, assistant_response_str)
    """
    env     = TeacherWorkspaceEnvironment()
    obs     = env.reset(task_name=task_name)
    history = []
    pairs   = []   # (user_prompt, assistant_json_str)
    consec_fails = 0

    for step in range(1, MAX_STEPS[task_name] + 1):
        if obs.done:
            break

        user_prompt = obs_to_prompt(obs, step, history)
        messages    = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_prompt},
        ]
        raw = call_api(messages, temperature=0.2)
        if raw is None:
            consec_fails += 1
            if consec_fails >= 3:
                break
            continue
        consec_fails = 0

        action_dict = parse_action(raw)
        if action_dict is None:
            history.append(f'Step {step}: parse error')
            continue

        # Store the pair BEFORE executing (so we have clean prompt→response)
        pairs.append((user_prompt, raw.strip()))

        # __done__ signal — stop collecting
        if action_dict['tool_name'] == '__done__':
            break

        try:
            action = TeacherAction(tool_name=action_dict['tool_name'],
                                   params=action_dict.get('params', {}))
            obs    = env.step(action)
            history.append(
                f"Step {step}: {action_dict['tool_name']} "
                f"reward={obs.reward:+.2f} {'OK' if obs.success else 'FAIL'}"
            )
        except Exception as e:
            history.append(f'Step {step}: exception {e}')

    return pairs, env.grade()


# ── Run data collection ──────────────────────────────────────────────────────
all_sft_records = []
baselines       = {}   # store 72B baseline grades per task

print(f'Collecting SFT data from {TEACHER_MODEL}')
print(f'Episodes per task: {EPISODES_PER_TASK} | Grade threshold: {GRADE_THRESHOLD}')
print('-' * 60)

for task in TASK_NAMES:
    kept   = 0
    grades = []
    print(f'\nTask: {task}')

    for ep in range(EPISODES_PER_TASK):
        pairs, grade = run_teacher_episode(task)
        grades.append(grade)
        status = 'KEPT' if grade >= GRADE_THRESHOLD else 'skip'
        print(f'  ep {ep+1:02d}/{EPISODES_PER_TASK}  grade={grade:.2f}  {status}  ({len(pairs)} steps)')

        if grade >= GRADE_THRESHOLD:
            for (user_p, asst_r) in pairs:
                all_sft_records.append({
                    'task':      task,
                    'messages': [
                        {'role': 'system',    'content': SYSTEM_PROMPT},
                        {'role': 'user',      'content': user_p},
                        {'role': 'assistant', 'content': asst_r},
                    ]
                })
            kept += 1

    baselines[task] = sum(grades) / len(grades)
    print(f'  => kept {kept}/{EPISODES_PER_TASK} episodes | baseline avg grade = {baselines[task]:.3f}')

# Save to disk
with open(SFT_DATA_PATH, 'w') as f:
    for rec in all_sft_records:
        f.write(json.dumps(rec) + '\n')

print(f'\nTotal SFT training pairs: {len(all_sft_records)}')
print(f'Saved to {SFT_DATA_PATH}')
print(f'Baselines (Qwen 72B): {baselines}')

## 7. Load Qwen3-4B in 4-bit with LoRA

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN  = 2048
STUDENT_MODEL = 'unsloth/Qwen3-4B-unsloth-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=STUDENT_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Qwen3-4B loaded with LoRA')
model.print_trainable_parameters()

## 8. Evaluate Raw 4B Baseline (before any training)

Runs locally on T4 — no API needed.

In [ ]:
def run_local_episode(task_name: str, mdl, tok, temperature: float = 0.3):
    """Run one episode with a local model. Returns (pairs, final_score, grade)."""
    env     = TeacherWorkspaceEnvironment()
    obs     = env.reset(task_name=task_name)
    history = []
    pairs   = []

    FastLanguageModel.for_inference(mdl)

    for step in range(1, MAX_STEPS[task_name] + 1):
        if obs.done:
            break

        user_prompt = obs_to_prompt(obs, step, history)
        messages    = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_prompt},
        ]
        input_ids = tok.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors='pt'
        ).to('cuda')

        with torch.no_grad():
            out = mdl.generate(
                input_ids,
                max_new_tokens=256,
                temperature=temperature,
                do_sample=True,
                pad_token_id=tok.eos_token_id,
            )
        raw = tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

        full_prompt = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        pairs.append((full_prompt, raw))

        action_dict = parse_action(raw)
        if action_dict is None:
            history.append(f'Step {step}: parse error')
            continue
        if action_dict['tool_name'] == '__done__':
            break

        try:
            action = TeacherAction(tool_name=action_dict['tool_name'],
                                   params=action_dict.get('params', {}))
            obs    = env.step(action)
            history.append(
                f"Step {step}: {action_dict['tool_name']} "
                f"reward={obs.reward:+.2f} {'OK' if obs.success else 'FAIL'}"
            )
        except Exception as e:
            history.append(f'Step {step}: exception {e}')

    return pairs, env.final_score(), env.grade()


print('=== RAW Qwen3-4B BASELINE (before training) ===')
raw_4b_grades = {}
for task in TASK_NAMES:
    task_grades = []
    for ep in range(3):
        _, _, g = run_local_episode(task, model, tokenizer, temperature=0.1)
        task_grades.append(g)
        print(f'  {task} ep{ep+1}: grade={g:.3f}')
    raw_4b_grades[task] = sum(task_grades) / len(task_grades)
    print(f'  => avg = {raw_4b_grades[task]:.3f}\n')

print('Raw 4B baseline:', raw_4b_grades)

## 9. SFT Training on Qwen 72B Trajectories

Uses HF TRL `SFTTrainer`. Trains on the `sft_data.jsonl` collected in cell 6.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel

# Load SFT dataset
with open(SFT_DATA_PATH) as f:
    records = [json.loads(l) for l in f]

# Convert to text format for SFTTrainer
def format_record(rec):
    return {'text': tokenizer.apply_chat_template(
        rec['messages'], tokenize=False, add_generation_prompt=False
    )}

sft_dataset = Dataset.from_list([format_record(r) for r in records])
print(f'SFT dataset size: {len(sft_dataset)} examples')
print(f'Sample: {sft_dataset[0]["text"][:200]}...')

# SFT config — conservative for T4
sft_config = SFTConfig(
    output_dir          = 'sft_checkpoints',
    num_train_epochs    = 2,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate       = 2e-4,
    warmup_ratio        = 0.1,
    lr_scheduler_type   = 'cosine',
    fp16                = not torch.cuda.is_bf16_supported(),
    bf16                = torch.cuda.is_bf16_supported(),
    logging_steps       = 5,
    save_steps          = 50,
    max_seq_length      = MAX_SEQ_LEN,
    dataset_text_field  = 'text',
    report_to           = 'wandb',
)

FastLanguageModel.for_training(model)

sft_trainer = SFTTrainer(
    model     = model,
    tokenizer = tokenizer,
    train_dataset = sft_dataset,
    args      = sft_config,
)

print('Starting SFT training...')
sft_result = sft_trainer.train()
sft_loss_history = [x['loss'] for x in sft_trainer.state.log_history if 'loss' in x]
print(f'SFT complete. Final loss: {sft_loss_history[-1]:.4f}')

## 10. Evaluate After SFT

In [ ]:
print('=== SFT Qwen3-4B EVALUATION ===')
sft_grades = {}
for task in TASK_NAMES:
    task_grades = []
    for ep in range(3):
        _, _, g = run_local_episode(task, model, tokenizer, temperature=0.1)
        task_grades.append(g)
        print(f'  {task} ep{ep+1}: grade={g:.3f}')
    sft_grades[task] = sum(task_grades) / len(task_grades)
    print(f'  => avg = {sft_grades[task]:.3f}\n')

print('\n--- SFT improvement over raw 4B ---')
for task in TASK_NAMES:
    delta = sft_grades[task] - raw_4b_grades[task]
    print(f'  {task:<25}  raw={raw_4b_grades[task]:.3f}  sft={sft_grades[task]:.3f}  delta={delta:+.3f}')

## 11. GRPO Training (starting from SFT checkpoint)

Uses environment reward signal (`env.final_score()`) to further improve the SFT model.
Group size = 4 rollouts per update step.
Advantages normalized within each group.

In [ ]:
from torch.optim import AdamW

GRPO_STEPS        = 150
EPISODES_PER_STEP = 4
LR                = 3e-5   # lower LR than SFT — we're refining not relearning

optimizer  = AdamW(model.parameters(), lr=LR)
grade_log  = []
reward_log = []
loss_log   = []
task_log   = []

print(f'Starting GRPO: {GRPO_STEPS} steps, {EPISODES_PER_STEP} episodes/step')
print('-' * 60)

for step in range(GRPO_STEPS):
    task_name = TASK_NAMES[step % len(TASK_NAMES)]
    task_log.append(task_name)

    all_pairs = []
    scores    = []
    grades    = []

    for _ in range(EPISODES_PER_STEP):
        pairs, fscore, gscore = run_local_episode(
            task_name, model, tokenizer, temperature=0.8
        )
        all_pairs.append(pairs)
        scores.append(fscore)
        grades.append(gscore)

    scores_t   = torch.tensor(scores, dtype=torch.float32)
    advantages = (scores_t - scores_t.mean()) / (scores_t.std() + 1e-8)

    FastLanguageModel.for_training(model)
    optimizer.zero_grad()
    total_loss = torch.tensor(0.0, requires_grad=True, device='cuda')

    for ep_idx, (pairs, adv) in enumerate(zip(all_pairs, advantages)):
        if not pairs:
            continue
        for (prompt_text, response_text) in pairs:
            full_text  = prompt_text + response_text
            enc        = tokenizer(full_text, return_tensors='pt',
                                   truncation=True, max_length=MAX_SEQ_LEN).to('cuda')
            prompt_enc = tokenizer(prompt_text, return_tensors='pt',
                                   truncation=True, max_length=MAX_SEQ_LEN).to('cuda')
            prompt_len = prompt_enc['input_ids'].shape[1]

            out       = model(**enc, labels=enc['input_ids'])
            logits    = out.logits[:, prompt_len-1:-1, :]
            labels    = enc['input_ids'][:, prompt_len:]
            log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
            token_lp  = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
            seq_lp    = token_lp.mean()
            total_loss = total_loss + (-adv.to('cuda') * seq_lp)

    total_loss = total_loss / (EPISODES_PER_STEP + 1e-8)
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    mean_score = float(scores_t.mean())
    mean_grade = sum(grades) / len(grades)
    loss_val   = float(total_loss.detach())

    reward_log.append(mean_score)
    grade_log.append(mean_grade)
    loss_log.append(loss_val)

    if step % 10 == 0:
        print(f'Step {step:3d} | task={task_name:<20} | '
              f'grade={mean_grade:.3f} | score={mean_score:.3f} | loss={loss_val:.4f}')

    try:
        wandb.log({'grpo_step': step, 'grpo_grade': mean_grade,
                   'grpo_score': mean_score, 'grpo_loss': loss_val})
    except:
        pass

print('GRPO training complete!')

## 12. Plot All Curves

In [ ]:
def moving_avg(arr, w=8):
    if len(arr) < w:
        return arr
    return np.convolve(arr, np.ones(w)/w, mode='valid').tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# SFT loss
ax = axes[0, 0]
ax.plot(sft_loss_history, alpha=0.4, color='purple', label='raw')
ax.plot(moving_avg(sft_loss_history), color='purple', linewidth=2, label='moving avg')
ax.set_title('SFT Training Loss', fontsize=12)
ax.set_xlabel('SFT Step')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# GRPO grade
ax = axes[0, 1]
ax.plot(grade_log, alpha=0.3, color='steelblue', label='raw')
ax.plot(moving_avg(grade_log), color='steelblue', linewidth=2, label='moving avg')
ax.set_title('GRPO Grade Score (env.grade())', fontsize=12)
ax.set_xlabel('GRPO Step')
ax.set_ylabel('Score (0-1)')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

# GRPO final score
ax = axes[1, 0]
ax.plot(reward_log, alpha=0.3, color='darkorange', label='raw')
ax.plot(moving_avg(reward_log), color='darkorange', linewidth=2, label='moving avg')
ax.set_title('GRPO Final Score (grade + efficiency)', fontsize=12)
ax.set_xlabel('GRPO Step')
ax.set_ylabel('Score (0-1)')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

# GRPO loss
ax = axes[1, 1]
ax.plot(loss_log, alpha=0.3, color='crimson', label='raw')
ax.plot(moving_avg(loss_log), color='crimson', linewidth=2, label='moving avg')
ax.set_title('GRPO Policy Loss', fontsize=12)
ax.set_xlabel('GRPO Step')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Teacher Workspace Agent — SFT → GRPO Training\n(Qwen3-4B distilled from Qwen2.5-72B)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved reward_curve.png')

## 13. Final Before / After / After Table

In [ ]:
print('=== FINAL EVALUATION: GRPO model ===')
grpo_grades = {}
for task in TASK_NAMES:
    task_grades = []
    for ep in range(3):
        _, _, g = run_local_episode(task, model, tokenizer, temperature=0.1)
        task_grades.append(g)
        print(f'  {task} ep{ep+1}: grade={g:.3f}')
    grpo_grades[task] = sum(task_grades) / len(task_grades)
    print(f'  => avg = {grpo_grades[task]:.3f}\n')

# Summary table
header = f'{"Task":<25} {"72B Teacher":>12} {"4B Raw":>10} {"4B SFT":>10} {"4B GRPO":>10}'
print('\n' + '=' * 72)
print(header)
print('-' * 72)
for task in TASK_NAMES:
    t72  = baselines.get(task, 0)
    t4r  = raw_4b_grades.get(task, 0)
    t4s  = sft_grades.get(task, 0)
    t4g  = grpo_grades.get(task, 0)
    print(f'{task:<25} {t72:>12.3f} {t4r:>10.3f} {t4s:>10.3f} {t4g:>10.3f}')
print('-' * 72)
avg72 = sum(baselines.values())     / len(TASK_NAMES)
avg4r = sum(raw_4b_grades.values()) / len(TASK_NAMES)
avg4s = sum(sft_grades.values())    / len(TASK_NAMES)
avg4g = sum(grpo_grades.values())   / len(TASK_NAMES)
print(f'{"Average":<25} {avg72:>12.3f} {avg4r:>10.3f} {avg4s:>10.3f} {avg4g:>10.3f}')
print('=' * 72)
print(f'\nSFT improvement  over raw 4B  : {avg4s - avg4r:+.3f}')
print(f'GRPO improvement over SFT 4B  : {avg4g - avg4s:+.3f}')
print(f'Total improvement over raw 4B : {avg4g - avg4r:+.3f}')

## 14. Save & Push to HuggingFace Hub

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted

model.save_pretrained('teacher-agent-qwen3-4b-lora')
tokenizer.save_pretrained('teacher-agent-qwen3-4b-lora')
print('Saved locally')

model.push_to_hub('amydosomething/teacher-workspace-agent-qwen3-4b')
tokenizer.push_to_hub('amydosomething/teacher-workspace-agent-qwen3-4b')
print('Pushed to HF Hub: amydosomething/teacher-workspace-agent-qwen3-4b')

## 15. Commit reward_curve.png to GitHub repo

In [ ]:
!cp reward_curve.png /content/teacher-workspace-env/reward_curve.png
%cd /content/teacher-workspace-env
!git config user.email 'you@example.com'
!git config user.name 'amydosomething'
!git add reward_curve.png train_teacher_sft_grpo.ipynb
!git commit -m 'Add SFT+GRPO training notebook and reward curve'
# Uncomment and fill your token:
# !git push https://<YOUR_GITHUB_TOKEN>@github.com/amydosomething/teacher-workspace-env.git